In [ ]:
-- Step 1: Enrich tweets with political metadata
-- Join tweet table → legislators DB by screen_name
# Enrichment
SELECT
    t.tweet_id,
    t.user_id,
    t.text,
    t.created_at,
    t.retweet_count,
    t.favorite_count,
    l.party,
    l.chamber,
    l.state,
    YEAR(t.created_at)  AS year,
    MONTH(t.created_at) AS month
FROM tweets t
LEFT JOIN legislators l
    ON LOWER(t.screen_name) = LOWER(l.twitter)
WHERE l.party IN ('Republican','Democrat')
ORDER BY t.created_at;

-- Result: 924,517 matched rows (74.8% of 1,243,370)
-- 138 members (302,898 tweets) retained but excluded
-- from party/chamber comparisons — declared exclusion

-- Lobbying Leverage Score (LLS) per member
-- Strips viral noise; uses median not mean
# SQL Query
SELECT
    m.name,
    m.party,
    m.chamber,
    m.state,
    MEDIAN(t.retweet_count)                           AS median_rt,
    COUNT(t.tweet_id)                                 AS tweet_count,
    CASE WHEN m.chamber='Senate' THEN 1.8 ELSE 1.0 END AS chamber_mult,
    ROUND(
      MEDIAN(t.retweet_count)
      * LN(COUNT(t.tweet_id) + 1)
      * (CASE WHEN m.chamber='Senate' THEN 1.8 ELSE 1.0 END),
    2) AS LLS
FROM tweets t
JOIN legislators m ON t.user_id = m.user_id
WHERE m.party IN ('Republican','Democrat')
GROUP BY m.name, m.party, m.chamber, m.state
HAVING tweet_count >= 10
ORDER BY LLS DESC
LIMIT 20;

-- Tiers: LLS > 20 = Priority 1 | 10-20 = Tier 2 | <10 = Monitor